In [ ]:
# This Python 3 environment comes with many helpful analytics libraries installed
# It is defined by the kaggle/python Docker image: https://github.com/kaggle/docker-python
# For example, here's several helpful packages to load

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)

# Input data files are available in the read-only "../input/" directory
# For example, running this (by clicking run or pressing Shift+Enter) will list all files under the input directory

import os
for dirname, _, filenames in os.walk('/kaggle/input'):
    for filename in filenames:
        print(os.path.join(dirname, filename))

# You can write up to 20GB to the current directory (/kaggle/working/) that gets preserved as output when you create a version using "Save & Run All" 
# You can also write temporary files to /kaggle/temp/, but they won't be saved outside of the current session

In [ ]:
!pip uninstall -y pinecone-client

In [1]:
!pip install pinecone langchain pypdf sentence-transformers tiktoken

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 745.9/745.9 kB 19.0 MB/s eta 0:00:0000:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 280.9/280.9 kB 14.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 65.5/65.5 kB 2.9 MB/s eta 0:00:00
  Attempting uninstall: packaging
    Found existing installation: packaging 26.0rc2
    Uninstalling packaging-26.0rc2:
      Successfully uninstalled packaging-26.0rc2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
bigframes 2.26.0 requires google-cloud-bigquery-storage<3.0.0,>=2.30.0, which is not installed.
google-colab 1.0.0 requires google-auth==2.38.0, but you have google-auth 2.47.0 which is incompatible.
google-colab 1.0.0 requires jupyter-server==2.14.0, but you have jupyter-server 2.12.5 which is incompatible.
google-cola

In [2]:
import os
from pypdf import PdfReader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from sentence_transformers import SentenceTransformer
import pinecone

2026-02-07 09:29:10.322161: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1770456550.565030      55 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1770456550.633272      55 cuda_blas.cc:1407] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
W0000 00:00:1770456551.153153      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770456551.153193      55 computation_placer.cc:177] computation placer already registered. Please check linkage and avoid linking the same target more than once.
W0000 00:00:1770456551.153202      55 computation_placer.cc:177] computation placer alr

In [3]:
pdf_path = "/kaggle/input/bhagwat-gita-english/bhagavad-gita-in-english-source-file.pdf"

reader = PdfReader(pdf_path)

text = ""
for page in reader.pages:
    text += page.extract_text() + "\n"

print(len(text))

144097


In [4]:
text = text.replace("\n\n", "\n")
text = text.replace("  ", " ")

In [5]:
start_marker = "INTRODUCTION"

start_idx = text.find(start_marker)
text_main = text[start_idx:]

print(text_main[:500])

INTRODUCTION ...................................................... 1 
1. Arjuna’s Dilemma .................................................. 3 
2. Spiritual knowledge................................................ 3 
The spirit is eternal, body is transitory ........................ 4 
Death and Reincarnation of the soul .......................... 4 
Duty of a warrior ......................................................... 5 
Importance of Karma-yoga, the selfless action .......... 5 
Materi


In [6]:
import re

chapter_pattern = r"CHAPTER\s+\d+"
chapters = re.split(chapter_pattern, text_main)

chapter_headers = re.findall(chapter_pattern, text_main)

print(len(chapters), len(chapter_headers))

19 18


In [7]:
chapter_blocks = list(zip(chapter_headers, chapters[1:]))

In [8]:
def extract_title(block):
    lines = block.strip().split("\n")
    return lines[0][:120]

In [9]:
pattern = r"\(\d+\.\d+\)"

parts = re.split(pattern, text)
markers = re.findall(pattern, text)

chunks = []

for marker, content in zip(markers, parts[1:]):
    
    chap, verse = marker.strip("()").split(".")
    
    chunk_text = content.strip()
    
    chunks.append({
        "id": f"ch{chap}_v{verse}",
        "chapter": int(chap),
        "verse": int(verse),
        "text": chunk_text
    })

print("Total verse chunks:", len(chunks))
print(chunks[2])

Total verse chunks: 533
{'id': 'ch2_v10', 'chapter': 2, 'verse': 10, 'text': 'Teachings of the Gita begins \n Important verses are high lighted. First time readers should \nread and understand these verses first. \n4              International Gita Society \n \n \nThe Supreme Lord said: You grieve for those who are not wor-\nthy of grief; and yet speak words of wisdom. The wise grieve nei-\nther for the living nor for the dead.'}


In [10]:
def subchunk(text, max_chars=900):
    return [text[i:i+max_chars] for i in range(0, len(text), max_chars)]

final_chunks = []

for c in chunks:
    pieces = subchunk(c["text"])
    
    for i, p in enumerate(pieces):
        final_chunks.append({
            "id": f'{c["id"]}_p{i}',
            "chapter": c["chapter"],
            "verse": c["verse"],
            "text": p
        })

print("Final chunks:", len(final_chunks))

Final chunks: 541


In [ ]:
import json

with open("gita_chunks.json", "w") as f:
    json.dump(final_chunks, f, indent=2)

In [12]:
from sentence_transformers import SentenceTransformer
model = SentenceTransformer("sentence-transformers/all-mpnet-base-v2")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/571 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/239 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [15]:
texts = [c["text"] for c in final_chunks]

embeddings = model.encode(
    texts,
    batch_size=32,
    show_progress_bar=True
)

Batches:   0%|          | 0/17 [00:00<?, ?it/s]

In [16]:
from pinecone import Pinecone
from kaggle_secrets import UserSecretsClient
import os

# Load secret
user_secrets = UserSecretsClient()
pinecone_key = user_secrets.get_secret("PINECONE_API_KEY")

# Optional: set env var (if you want)
os.environ["PINECONE_API_KEY"] = pinecone_key

# Init Pinecone
pc = Pinecone(api_key=pinecone_key)

# Connect index
index = pc.Index("bgeeta")

# Check stats
print(index.describe_index_stats())

{'_response_info': {'raw_headers': {'connection': 'keep-alive',
                                    'content-length': '185',
                                    'content-type': 'application/json',
                                    'date': 'Sat, 07 Feb 2026 09:34:36 GMT',
                                    'grpc-status': '0',
                                    'server': 'envoy',
                                    'x-envoy-upstream-service-time': '70',
                                    'x-pinecone-request-latency-ms': '69',
                                    'x-pinecone-response-duration-ms': '71'}},
 'dimension': 768,
 'index_fullness': 0.0,
 'memoryFullness': 0.0,
 'metric': 'cosine',
 'namespaces': {'__default__': {'vector_count': 541}},
 'storageFullness': 0.0,
 'total_vector_count': 541,
 'vector_type': 'dense'}


In [18]:
vectors = []

for chunk, emb in zip(final_chunks, embeddings):
    
    vectors.append((
        chunk["id"],
        emb.tolist(),
        {
            "chapter": chunk["chapter"],
            "verse": chunk["verse"],
            "text": chunk["text"][:500]  # small preview only
        }
    ))

# batch upload
batch_size = 100

for i in range(0, len(vectors), batch_size):
    index.upsert(vectors[i:i+batch_size])

In [17]:
query = "How to control senses?"

q_emb = model.encode([query])[0]

res = index.query(
    vector=q_emb.tolist(),
    top_k=5,
    include_metadata=True
)

for match in res["matches"]:
    print(match["score"])
    print(match["metadata"])
    print()

0.576464653
{'chapter': 2, 'text': 'The contacts of the senses with the \nsense objects give rise to the feelings of heat and cold, pain and \npleasure. They are transitory and impermanent. Therefore, learn \nto endure them, O Arjuna,', 'verse': 13}

0.559350073
{'chapter': 3, 'text': 'Two stumbling blocks on the path \nLikes and dislikes for sense objects remain in the senses. One \nshould not come under the control of these two, because they are, \nindeed, two major stumbling blocks on one’s path of Self -realiza-\ntion.', 'verse': 33}

0.531122267
{'chapter': 2, 'text': 'Dangers of unrestrained senses \nRestless senses, O Arjuna, forcibly carry away the mind of \neven a wise person striving for perfection.', 'verse': 59}

0.509515822
{'chapter': 2, 'text': 'When one can com-\npletely withdraw the senses from its sense objects— as a tortoise \nwithdraws its limbs into the shell for protection from dangers— then \nthe intellect of such a person is considered steady.', 'verse': 57}

0.

MULTI HOP RAG

In [20]:
!pip install groq

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 138.3/138.3 kB 6.8 MB/s eta 0:00:00


In [22]:
from groq import Groq
from kaggle_secrets import UserSecretsClient

# Load secrets from Kaggle
user_secrets = UserSecretsClient()
GROQ_API_KEY = user_secrets.get_secret("GROQ_API_KEY")
PINECONE_API_KEY = user_secrets.get_secret("PINECONE_API_KEY")

# Create Groq client with secret directly
client = Groq(api_key=GROQ_API_KEY)


def decompose_query(query):
    prompt = f"""
Break this question into reasoning sub-questions for scripture search.

Question: {query}

Return exactly 3 search questions only.
"""

    r = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role": "user", "content": prompt}],
        temperature=0.2
    )

    # Clean split
    lines = r.choices[0].message.content.strip().split("\n")

    # Remove empty lines
    return [l.strip("-• ").strip() for l in lines if l.strip()]

In [46]:
def final_llm_answer(query, context):

    prompt = f"""
Tum ek spiritual RAG assistant ho jo Bhagavad Gita context se answer deta hai.

RULES:
- Answer sirf diye gaye context se dena
- Agar context me answer nahi hai → bolo "context me clear answer nahi mila"
- Pure Hinglish me likho
- user se doston ki tarhan bat kro, ek guide ki trhan nai
- prompts me age jate jate user ke is tarha mitr ban jao ki use pata bhi na chale
- tumhe har chiz ki refrence deni ha with shloka
- show interest in solving user's problem ask questions about the issue
- Tone simple aur samjhane wala ho

CONTEXT:
{context}

USER QUESTION:
{query}

HINGLISH ANSWER:
"""

    r = client.chat.completions.create(
        model="llama-3.3-70b-versatile",
        messages=[{"role":"user","content":prompt}]
    )

    return r.choices[0].message.content

In [23]:
def retrieve_chunks(q, k=4):
    q_emb = model.encode([q])[0]

    res = index.query(
        vector=q_emb.tolist(),
        top_k=k,
        include_metadata=True
    )

    return [m["metadata"]["text"] for m in res["matches"]]

In [39]:
def multihop_rag(query, history=None):

    subqs = decompose_query(query)

    all_evidence = []

    for sq in subqs:
        chunks = retrieve_chunks(sq)
        all_evidence.extend(chunks)

    try:
        all_evidence = rerank(query, all_evidence)
    except:
        pass

    rag_context = "\n".join(all_evidence)

    # ✅ build memory text
    memory_text = ""
    if history:
        memory_text = "\n".join([
            f"User: {h[0]}\nAI: {h[1]}"
            for h in history[-5:]   # last 5 turns only
        ])

    full_context = f"""
CHAT MEMORY:
{memory_text}

SCRIPTURE CONTEXT:
{rag_context}
"""

    return final_llm_answer(query, full_context)

In [26]:
from sentence_transformers import CrossEncoder

reranker = CrossEncoder('cross-encoder/ms-marco-MiniLM-L-6-v2')

def rerank(query, passages):
    pairs = [[query, p] for p in passages]
    scores = reranker.predict(pairs)
    ranked = sorted(zip(passages, scores), key=lambda x: x[1], reverse=True)
    return [p for p,_ in ranked[:5]]

config.json:   0%|          | 0.00/794 [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/90.9M [00:00<?, ?B/s]

tokenizer_config.json: 0.00B [00:00, ?B/s]

vocab.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/132 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

In [27]:
!pip install gradio

huggingface/tokenizers: The current process just got forked, after parallelism has already been used. Disabling parallelism to avoid deadlocks...
To disable this warning, you can either:
	- Avoid using `tokenizers` before the fork if possible
	- Explicitly set the environment variable TOKENIZERS_PARALLELISM=(true | false)


     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 68.6/68.6 kB 4.0 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 444.8/444.8 kB 18.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 2.0/2.0 MB 60.4 MB/s eta 0:00:00:00:01
  Attempting uninstall: pydantic-core
    Found existing installation: pydantic_core 2.41.5
    Uninstalling pydantic_core-2.41.5:
      Successfully uninstalled pydantic_core-2.41.5
  Attempting uninstall: pydantic
    Found existing installation: pydantic 2.12.5
    Uninstalling pydantic-2.12.5:
      Successfully uninstalled pydantic-2.12.5
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-adk 1.22.1 requires google-cloud-bigquery-storage>=2.0.0, which is not installed.
fastai 2.8.4 requires fastcore<1.9,>=1.8.0, but you have fastcore 1.11.3 which is incompatible.


In [47]:
import gradio as gr

def chat_fn(message, history):
    answer = multihop_rag(message, history)
    return answer

gr.ChatInterface(chat_fn).launch()

/usr/local/lib/python3.12/dist-packages/gradio/chat_interface.py:347: UserWarning: The 'tuples' format for chatbot messages is deprecated and will be removed in a future version of Gradio. Please set type='messages' instead, which uses openai-style 'role' and 'content' keys.
  self.chatbot = Chatbot(


* Running on local URL:  http://127.0.0.1:7867
It looks like you are running Gradio on a hosted Jupyter notebook, which requires `share=True`. Automatically setting `share=True` (you can turn this off by setting `share=False` in `launch()` explicitly).

* Running on public URL: https://7301dc9d30f66bc2a6.gradio.live

This share link expires in 1 week. For free permanent hosting and GPU upgrades, run `gradio deploy` from the terminal in the working directory to deploy to Hugging Face Spaces (https://huggingface.co/spaces)
